# 법정동(전국) → DuckDB 적재

공공데이터포털 `StanReginCd/getStanReginCdList` API. 응답 구조가 타 API와 달라(배열 기반) 별도 파일로 분리.

- 대상: 전국 법정동 (4만건+ 예상, 재사용 목적)
- 테이블: `region` (PK=region_cd 10자리)
- self-ref: `locathigh_cd` → 상위 행정구역

In [ ]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
# 락 점유 최소화: db_open() 컨텍스트 매니저로 짧게 연결하고 try/finally로 반드시 닫음.
# 예외가 나도 커넥션이 해제되므로 여러 노트북 동시 사용 가능.

import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저.

    - read_only=True: 여러 커넥션 동시 허용
    - read_only=False: 쓰기 락. 블록 내부에서만 점유
    - try/finally: 예외 발생 시에도 con.close() 보장
    """
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

In [ ]:
# ============================================================
# 셀 2 - StanReginCd 전용 Pydantic 모델
# ============================================================
# ⚠️ 이 API는 다른 공공데이터 API와 응답 구조가 다름!
#
# 일반 공공데이터 API:
#   { "Response": { "header": {...}, "body": { "items": { "item": [...] } } } }
#
# StanReginCd (법정동 API):
#   { "StanReginCd": [
#         { "head": [ { totalCount, numOfRows, pageNo, RESULT: {resultCode, resultMsg} } ] },
#         { "row":  [ {...}, {...}, ... ] }
#   ] }
#
# StanReginCd가 길이-2 배열이고 [0]에 head, [1]에 row가 나뉘어 있는 특이 구조.
# → PublicDataResponse 제네릭 재사용 불가, 전용 모델 필요.

from pydantic import BaseModel, Field
from typing import Optional


class StanReginResult(BaseModel):
    """head 안의 RESULT 필드. 정상 코드는 'INFO-000'."""
    resultCode: str
    resultMsg: str


class StanReginHead(BaseModel):
    """head 블록 원소. 페이지 메타 + 결과 코드."""
    totalCount: Optional[int] = None
    numOfRows: Optional[str] = None       # 문자열로 옴 (API 특이사항)
    pageNo: Optional[str] = None
    type: Optional[str] = None
    RESULT: Optional[StanReginResult] = None


class RegionRow(BaseModel):
    """row 블록 원소 = 법정동 한 건.

    region_cd는 10자리 계층 구조 코드:
      [sido_cd(2)][sgg_cd(3)][umd_cd(3)][ri_cd(2)]
    예: "1159010100" = 11(서울) + 590(동작구) + 101(노량진동) + 00

    locathigh_cd는 상위 행정구역 코드 (self-reference).
      "1159010100" (노량진동) → locathigh_cd = "1159000000" (동작구)
    """
    region_cd: str          # 10자리 전체 코드
    sido_cd: str            # 2자리 시도 (11=서울, 26=부산, ...)
    sgg_cd: str             # 3자리 시군구
    umd_cd: str             # 3자리 읍면동
    ri_cd: str              # 2자리 리 (대부분 00)
    locatjumin_cd: Optional[str] = None   # 주민용 코드
    locatjijuk_cd: Optional[str] = None   # 지적용 코드
    locatadd_nm: str        # 풀네임 "서울특별시 동작구 노량진동"
    locat_order: Optional[int] = None     # 순번
    locat_rm: Optional[str] = None        # 비고
    locathigh_cd: Optional[str] = None    # 상위 코드 (self-ref FK)
    locallow_nm: str        # 짧은 이름 "노량진동"
    adpt_de: Optional[str] = None         # 적용일자 (YYYYMMDD)


class StanReginBlock(BaseModel):
    """StanReginCd 배열의 각 원소. head 또는 row 둘 중 하나만 채워짐."""
    head: Optional[list[StanReginHead]] = None
    row: Optional[list[RegionRow]] = None


class StanReginResponse(BaseModel):
    """API 응답 루트. StanReginCd는 길이-2 배열."""
    StanReginCd: list[StanReginBlock]

    def get_head(self) -> StanReginHead:
        """head 블록의 첫 번째 메타 정보를 꺼냄."""
        for block in self.StanReginCd:
            if block.head:
                return block.head[0]
        raise ValueError("head block not found")

    def get_rows(self) -> list[RegionRow]:
        """row 블록의 법정동 리스트를 꺼냄. 없으면 빈 리스트."""
        for block in self.StanReginCd:
            if block.row:
                return block.row
        return []


print("모델 로드 완료")

In [ ]:
# ============================================================
# 셀 3 - 법정동 전용 fetch (페이지네이션)
# ============================================================
# 공공데이터 표준 응답과 달라서 fetch_all_pages 재사용 불가 → 별도 함수.
# 한 페이지 1000건, totalCount에 도달할 때까지 순회.

STAN_REGIN_URL = "https://apis.data.go.kr/1741000/StanReginCd/getStanReginCdList"


def fetch_all_regions(num_rows: int = 1000) -> list[RegionRow]:
    """전국 법정동 전체 수집. 약 4만건+ → 40~50페이지 예상."""
    all_rows: list[RegionRow] = []
    page = 1
    while True:
        # 이 API는 type=json (소문자) 씀. 타 API는 dataType=JSON → 규약 다름 주의
        params = {
            "serviceKey": SERVICE_KEY,
            "pageNo": page,
            "numOfRows": num_rows,
            "type": "json",
        }
        res = requests.get(STAN_REGIN_URL, params=params, timeout=30)
        res.raise_for_status()

        # Pydantic 검증 → 스키마 안전성 확보
        parsed = StanReginResponse.model_validate(res.json())

        # 결과 코드 확인. INFO-000이 정상 (공통 코드 체계 달라서 모두 허용 목록에 포함)
        head = parsed.get_head()
        if head.RESULT and head.RESULT.resultCode not in ("INFO-000", "00", "200"):
            raise RuntimeError(f"API error: {head.RESULT.resultCode} {head.RESULT.resultMsg}")

        rows = parsed.get_rows()
        all_rows.extend(rows)
        total = head.totalCount or 0
        print(f"  page {page}: +{len(rows)}건 (누적 {len(all_rows)}/{total})")

        # 종료 조건: 받은 게 없거나 누적이 totalCount 도달
        if not rows or len(all_rows) >= total:
            break

        page += 1
        time.sleep(0.15)   # API 부하 방지

    return all_rows

In [ ]:
# ============================================================
# 셀 4 - 전국 법정동 수집 실행
# ============================================================
# 주의: 약 4만건+ 수집 → 수십 페이지 순회에 10~20초 소요될 수 있음

region_rows = fetch_all_regions(num_rows=1000)

# Pydantic → dict → DataFrame
region_df = pd.DataFrame([r.model_dump() for r in region_rows])

print(f"\n총 {len(region_df)}건 수집")

# 시도별 분포로 데이터 완결성 확인 (정상이면 17개 시도에 고루 분포)
print(f"시도별 행수:")
print(region_df['sido_cd'].value_counts().sort_index())

print(f"\n샘플:")
print(region_df.head())

In [ ]:
# ============================================================
# 셀 5 - DuckDB 적재 (region 테이블)
# ============================================================
with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS region (
        region_cd      TEXT PRIMARY KEY,  -- 10자리 법정동 전체 코드
        sido_cd        TEXT,              -- 2자리 시도
        sgg_cd         TEXT,              -- 3자리 시군구
        umd_cd         TEXT,              -- 3자리 읍면동
        ri_cd          TEXT,              -- 2자리 리
        locatjumin_cd  TEXT,
        locatjijuk_cd  TEXT,
        locatadd_nm    TEXT,
        locat_order    INTEGER,
        locat_rm       TEXT,
        locathigh_cd   TEXT,              -- 상위 행정구역 (self-ref)
        locallow_nm    TEXT,
        adpt_de        TEXT
    )
    """)

    con.execute("DELETE FROM region")
    con.register("region_df_view", region_df)
    con.execute("""
    INSERT INTO region
    SELECT region_cd, sido_cd, sgg_cd, umd_cd, ri_cd,
           locatjumin_cd, locatjijuk_cd, locatadd_nm,
           locat_order, locat_rm, locathigh_cd, locallow_nm, adpt_de
    FROM region_df_view
    """)
    con.unregister("region_df_view")

    cnt = con.execute("SELECT COUNT(*) FROM region").fetchone()[0]

print(f"적재 완료: {cnt}건")

In [ ]:
# ============================================================
# 셀 6 - 검증 쿼리
# ============================================================
# read_only=True → 다른 노트북이 쓰기 중이어도 안전하게 읽음

with db_open(read_only=True) as con:
    print("=== 시도별 법정동 수 ===")
    print(con.execute("""
        SELECT sido_cd, COUNT(*) AS cnt,
               ANY_VALUE(CASE WHEN sgg_cd='000' AND umd_cd='000' THEN locatadd_nm END) AS sido_nm
        FROM region
        GROUP BY sido_cd ORDER BY sido_cd
    """).df())

    print("\n=== 서울 동작구의 동 목록 ===")
    print(con.execute("""
        SELECT region_cd, locatadd_nm, locallow_nm
        FROM region
        WHERE sido_cd='11' AND sgg_cd='590' AND umd_cd <> '000'
        ORDER BY region_cd
        LIMIT 20
    """).df())